In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools, algorithms

In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools, algorithms

NUM_DISKS = 5
NUM_PEGS = 3
POP_SIZE = 300
NGEN = 200
CXPB = 0.7
MUTPB = 0.2
TOURNAMENT_SIZE = 3
MAX_STEPS = 2 * NUM_DISKS * NUM_DISKS

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def init_individual():
    return [rnd.randint(0, NUM_PEGS - 1) for _ in range(MAX_STEPS)]

def evaluate(individual):
    pegs = [list(range(NUM_DISKS, 0, -1)), [], []]
    moves = 0
    for step in individual:
        if moves >= MAX_STEPS:
            break
        from_peg = step
        valid_to = [p for p in range(NUM_PEGS) if p != from_peg and (not pegs[from_peg] or (pegs[p] and pegs[from_peg][-1] < pegs[p][-1]) or not pegs[p])]
        if not valid_to:
            continue
        to_peg = rnd.choice(valid_to)
        disk = pegs[from_peg].pop()
        pegs[to_peg].append(disk)
        moves += 1
        if pegs[2] == list(range(NUM_DISKS, 0, -1)):
            return (moves,)
    penalty = MAX_STEPS + sum(len(p) for p in pegs[:2]) * 10
    return (penalty,)

def cx_two_point(ind1, ind2):
    size = min(len(ind1), len(ind2))
    cxpoint1 = rnd.randint(1, size - 1)
    cxpoint2 = rnd.randint(1, size - 1)
    if cxpoint2 < cxpoint1:
        cxpoint1, cxpoint2 = cxpoint2, cxpoint1
    ind1[cxpoint1:cxpoint2], ind2[cxpoint1:cxpoint2] = ind2[cxpoint1:cxpoint2], ind1[cxpoint1:cxpoint2]
    return ind1, ind2

def mut_uniform(individual, indpb=0.1):
    for i in range(len(individual)):
        if rnd.random() < indpb:
            individual[i] = rnd.randint(0, NUM_PEGS - 1)
    return (individual,)

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, init_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", cx_two_point)
toolbox.register("mutate", mut_uniform, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

In [ ]:
from deap import base, creator

creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
def evalTowerOfHanoi(individual, num_disks, source_peg=0, target_peg=2):
    pegs = [list(range(num_disks, 0, -1)), [], []]
    invalid_moves = 0

    for move in individual:
        if not isinstance(move, (tuple, list)) or len(move) != 2:
            invalid_moves += 1
            continue

        src, tgt = move
        if not (0 <= src < 3 and 0 <= tgt < 3):
            invalid_moves += 1
            continue

        if not pegs[src]:
            invalid_moves += 1
            continue

        disk = pegs[src][-1]

        if pegs[tgt] and pegs[tgt][-1] < disk:
            invalid_moves += 1
            continue

        pegs[src].pop()
        pegs[tgt].append(disk)

    correct = 0
    expected_disk = num_disks
    for disk in pegs[target_peg]:
        if disk == expected_disk:
            correct += 1
            expected_disk -= 1
        else:
            break

    return invalid_moves, -correct

In [ ]:
def cxTwoPointMoveList(ind1, ind2):
    size = min(len(ind1), len(ind2))
    if size < 2:
        return ind1, ind2

    cxpoint1 = rnd.randint(1, size)
    cxpoint2 = rnd.randint(1, size - 1)
    if cxpoint2 >= cxpoint1:
        cxpoint2 += 1
    else:
        cxpoint1, cxpoint2 = cxpoint2, cxpoint1

    ind1[cxpoint1:cxpoint2], ind2[cxpoint1:cxpoint2] = ind2[cxpoint1:cxpoint2], ind1[cxpoint1:cxpoint2]

    return ind1, ind2

In [ ]:
import random as rnd

def mut_hanoi(individual, indpb=0.1):
    """
    Mutates an individual by replacing a random move with a valid random move (src != dst).
    Assumes 3 pegs (0, 1, 2).
    """
    for i in range(len(individual)):
        if rnd.random() < indpb:
            src = rnd.randint(0, 2)
            dst = rnd.randint(0, 2)
            while src == dst:
                dst = rnd.randint(0, 2)
            individual[i] = (src, dst)
    return individual,

In [ ]:
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools

MAX_STEPS = 100
N_PEGS = 3

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def generate_valid_move():
    src = rnd.randint(0, N_PEGS - 1)
    dst = rnd.randint(0, N_PEGS - 1)
    while src == dst:
        dst = rnd.randint(0, N_PEGS - 1)
    return (src, dst)

toolbox = base.Toolbox()
toolbox.register("attr_move", generate_valid_move)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_move, n=MAX_STEPS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

N_DISKS = 5
N_PEGS = 3
MAX_STEPS = 2 * (2 ** N_DISKS)

def init_individual(icls):
    return icls([rnd.randint(0, N_PEGS - 1) for _ in range(MAX_STEPS * 2)])

toolbox.register("individual", init_individual, creator.Individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    pegs = [list(range(N_DISKS, 0, -1)), [], []]
    moves = [(individual[i], individual[i+1]) for i in range(0, len(individual), 2)]
    penalty = 0
    steps = 0
    
    for src, dst in moves:
        if steps >= MAX_STEPS:
            break
        if src == dst or not (0 <= src < N_PEGS) or not (0 <= dst < N_PEGS):
            penalty += 10
            continue
        if not pegs[src]:
            penalty += 5
            continue
        disk = pegs[src][-1]
        if pegs[dst] and pegs[dst][-1] < disk:
            penalty += 10
            continue
        
        pegs[src].pop()
        pegs[dst].append(disk)
        steps += 1
        
        if pegs[2] == list(range(N_DISKS, 0, -1)):
            return (steps + penalty,)

    remaining = sum(1 for d in range(1, N_DISKS + 1) if d not in pegs[2])
    penalty += remaining * 100
    penalty += (MAX_STEPS - steps) * 0.1
    
    return (penalty,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=0, up=N_PEGS-1, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
import random as rnd
import numpy as np
from deap import algorithms, tools

rnd.seed(42)
np.random.seed(42)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

hof = tools.HallOfFame(1)

population = toolbox.population(n=300)

algorithms.eaSimple(population, toolbox, cxpb=0.7, mutpb=0.2, ngen=100, 
                    stats=stats, halloffame=hof, verbose=True)

print("Best individual:", hof[0])
print("Best fitness:", hof[0].fitness.values)

In [ ]:
import numpy as np
from deap import tools

# Define statistics for each objective (index 0 and 1 of fitness.values)
stats_obj1 = tools.Statistics(lambda ind: ind.fitness.values[0])
stats_obj2 = tools.Statistics(lambda ind: ind.fitness.values[1])

# Register min, avg, max for both
for s in (stats_obj1, stats_obj2):
    s.register("min", np.min)
    s.register("avg", np.mean)
    s.register("max", np.max)

# Combine into MultiStatistics
mstats = tools.MultiStatistics(obj1=stats_obj1, obj2=stats_obj2)

In [ ]:
# Visualize Best Individual Solution
# Assumes 'best_ind' (list of moves) and 'N' (num disks) are defined in previous cells

def simulate_hanoi(individual, num_disks):
    pegs = [list(range(num_disks, 0, -1)), [], []]
    peg_names = ['A', 'B', 'C']
    
    def print_pegs(step, move=None):
        max_h = num_disks
        print(f"\nStep {step:3d}" + (f" | Move: {peg_names[move[0]]} -> {peg_names[move[1]]}" if move else " | INITIAL"))
        for level in range(max_h - 1, -1, -1):
            row = ""
            for p in range(3):
                if level < len(pegs[p]):
                    disk = pegs[p][level]
                    row += f"  {disk:2d}  "
                else:
                    row += "       "
            print(row)
        print("  A      B      C")
        print("-" * 25)

    print_pegs(0)
    
    valid = True
    for i, (frm, to) in enumerate(individual):
        if not pegs[frm]:
            print(f"ERROR Step {i+1}: Peg {peg_names[frm]} empty.")
            valid = False
            break
        disk = pegs[frm][-1]
        if pegs[to] and pegs[to][-1] < disk:
            print(f"ERROR Step {i+1}: Illegal move {disk} onto {pegs[to][-1]}.")
            valid = False
            break
        
        pegs[frm].pop()
        pegs[to].append(disk)
        print_pegs(i + 1, (frm, to))

    goal = [[], [], list(range(num_disks, 0, -1))]
    success = (pegs == goal)
    
    print("\n" + "="*30)
    print(f"FINAL STATE: {pegs}")
    print(f"GOAL STATE:  {goal}")
    print(f"VALID MOVES: {valid}")
    print(f"SOLVED:      {success}")
    print(f"TOTAL MOVES: {len(individual)} (Optimal: {2**num_disks - 1})")
    return success, valid

# Execute visualization
if 'best_ind' in globals() and 'N' in globals():
    simulate_hanoi(best_ind, N)
else:
    print("Variables 'best_ind' or 'N' not found. Run EA cells first.")